# Benchmark de Quantum Simulator en AMD ROCm

Este notebook ejecuta el benchmark del simulador statevector en la GPU de AMD (W7900D, RDNA 3, gfx1100) detectando el hardware en runtime y generando archivos de salida para registrar los resultados.

**Flujo:**
1. Detecta hardware (GPU, arquitectura GFX, versión ROCm).
2. Muestra comando de instalación de `pennylane-lightning-gpu` bajo demanda (si se quiere comparar).
3. Ejecuta barrido de benchmarks (n × batch × dtype).
4. Genera `benchmark_results_*.json` (datos brutos) y `BENCHMARK_SUMMARY_*.md` (resumen legible).

No compila nada de antemano. Todo es bajo demanda.

In [ ]:
# --- Configuración de rutas para importar desde entrega/simulador ---
import os, sys
from pathlib import Path

# Intentar rutas comunes en notebooks.amd.com
candidatas = [
    Path.cwd(),
    Path('/home/jesus/knowledge_database/obsidian_cerebro/02-PROYECTOS/hackathon-amd-act3/entrega'),
    Path('/mnt/notebooks'),
    Path('/workspace'),
]

entrega_dir = None
for c in candidatas:
    if (c / 'simulador' / 'qsim_torch.py').exists():
        entrega_dir = c
        break

if entrega_dir is None:
    # Último recurso: usar el directorio actual
    entrega_dir = Path.cwd()
    print(f"⚠️ No se encontró entrega/ automáticamente. Usando {entrega_dir}")
else:
    print(f"✓ Directorio de entrega: {entrega_dir}")

sys.path.insert(0, str(entrega_dir))
sys.path.insert(0, str(entrega_dir / 'simulador'))

# Verificar importaciones
from qsim_torch import Simulator, LAYERS
from hardware_wheels import detect_gpu_architecture, install_command, quick_summary, print_report

print(f"LAYERS por defecto: {LAYERS}")


In [ ]:
# --- Paso 1: Detección de hardware en runtime ---
print("=" * 70)
print("DETECCIÓN DE HARDWARE (runtime - no compila nada)")
print("=" * 70)

gpu = detect_gpu_architecture()
hw_summary = quick_summary()

print(f"GPU             : {gpu.name}")
print(f"Arquitectura    : {gpu.architecture}")
print(f"Versión ROCm    : {gpu.rocm_version}")
print(f"Es MI300 (gfx942): {gpu.is_mi300}")
print(f"Es gfx1100      : {gpu.is_gfx1100}")
print(f"Wheel precompilada: {'SÍ' if gpu.supports_precompiled else 'NO'}")
print()
print("Comando de instalación de pennylane-lightning-gpu (bajo demanda):")
print(f"  {install_command(gpu)}")
print()
print("Nota: el simulador PyTorch puro (qsim_torch.py) funciona sin pennylane-lightning-gpu.")
print("Solo instálalo si quieres comparar con lightning.amdgpu.")
print("=" * 70)

In [ ]:
# --- Paso 2: Ejecutar benchmark ---
from benchmark_notebook import run_benchmark

# Configuración para GPU (W7900D)
resultados = run_benchmark(
    device_str='cuda',        # 'cuda' para GPU AMD/ROCm, 'cpu' para CPU
    ns=[16, 20, 24, 26, 28],  # qubits a probar
    batches=[1, 256, 1024],   # tamaños de lote
    dtypes=None,              # [complex128, complex64] por defecto
    output_dir='.',           # guardar en el directorio actual
)

In [ ]:
# --- Paso 3: Mostrar archivos generados ---
import os
from pathlib import Path

out = Path(resultados['output_files']['json'])
md = Path(resultados['output_files']['markdown'])

print("=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)
print(f"JSON (datos brutos):   {out}  ({out.stat().st_size / 1024:.1f} KB)")
print(f"Markdown (resumen):    {md}  ({md.stat().st_size / 1024:.1f} KB)")
print()
print("Contenido del resumen:")
print("=" * 70)
print(md.read_text(encoding='utf-8'))


## Cómo descargar los resultados

En la interfaz de Jupyter de notebooks.amd.com:
1. Busca los archivos `benchmark_results_*.json` y `BENCHMARK_SUMMARY_*.md` en el explorador de archivos.
2. Haz clic derecho → Download para descargarlos a tu máquina local.
3. El JSON contiene todos los datos brutos (tiempos, través, memoria, estado de cada prueba).
4. El Markdown tiene el resumen ejecutivo listo para incluir en el informe del hackathon.

## Notas importantes

- La cuota de GPU en notebooks.amd.com es de 3 h/día por usuario. Apaga la sesión con "Turn-off Session" cuando termines.
- El simulador es device-agnostic: si no hay GPU disponible, corre en CPU automáticamente.
- No se afirma ventaja de velocidad sobre CPU; la GPU se justifica por memoria (lotes > 6 GiB).

In [ ]:
# Celda vacía para ejecutar código adicional si es necesario
# Ejemplo: ver GPU actual
import torch
if torch.cuda.is_available():
    print(f"GPU activa: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
    print(f"ROCm/HIP: {torch.version.hip}")
else:
    print("No hay GPU disponible. Ejecutando en CPU.")
